<table style="width:100%; font-size:11pt; border-collapse:collapse">
    <tr>
        <td colspan="2"
            style="border: 1px #0098cd solid;
                   background-color:#E6F4F9;
                   color:#0098CD;
                   text-align:center;
                   font-weight:bold;
                   padding:8px;">
            Universidad de Oriente
        </td>
    </tr>
    <tr>
        <td style="border: 1px #0098cd solid;
                   background-color:#E6F4F9;
                   color:#0098CD;
                   width:50%;
                   text-align:center;
                   padding:6px;">
                            Machine Learning
        </td>
        <td style="border: 1px #0098cd solid;
                   background-color:#E6F4F9;
                   color:#0098CD;
                   width:50%;
                   text-align:center;
                   padding:6px;">
            Clase 10 - Ejemplo 1 - Regresión de Ridge, Lasso y Red Elástica
        </td>
    </tr>
    <tr>
 

</table>

In [ ]:
import pandas as pd
import os
import tarfile
import urllib.request
import numpy as np
from pandas.core.common import flatten
from plotnine import *
from array import *
import scipy.stats as stats
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

## Ejemplo 1 - Parte 1

En este ejemplo se profundizará en los modelos de regresión penalizada. Cargar el dataset de ventas y publicidad, y realizar un modelo de regresión Ridge del volúmenes de las ventas con las variables restantes.

a) Con el parámetro de encogimiento $\alpha=10$

b) Con el parámetro de encogimiento $\alpha=100$

c) Con el parámetro de encogimiento $\alpha=1000$

d) Comparar los modelos con el modelo regresión lineal múltiple. ¿Qué se observa?

e) Realizar la validación cruzada para determinar el valor de $\alpha$ que proporciona un menor error cuadrático

f) Comparar el comportamiento de los coeficientes cuando $\alpha$ disminuye y cuando $\alpha$ tiende a hacerse más grande.

In [ ]:
#Cargar la base de datos

datos = pd.read_csv("https://raw.githubusercontent.com/rpizarrog/Analisis-Inteligente-de-datos/main/datos/Advertising_Web.csv")

In [ ]:
datos

In [ ]:
datos.info()

In [ ]:
respuesta = datos["Sales"].copy()
predictoras = datos.drop(["Unnamed: 0", "X", "Sales"], axis=1)

In [ ]:
#Estandarizar las variables
rom sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(predictoras)

predictoras_est = pd.DataFrame(scaler.transform(predictoras), columns = predictoras.columns)

In [ ]:
predictoras_est

In [ ]:
#Importar clase
from sklearn.linear_model import Ridge

#Ajustar el modelo ridge con alfa=10
ridge_reg = Ridge(alpha = 10, solver = "auto")
ridge_reg.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo
#Intercepto
print("El intercepto que se tiene es %s" %ridge_reg.intercept_)
#Coeficientes de regresion
coefs_ridge_a = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": ridge_reg.coef_})
coefs_ridge_a

In [ ]:
#Importar clase
from sklearn.linear_model import Ridge

#Ajustar el modelo ridge con alfa=100
ridge_reg_b = Ridge(alpha = 100, solver = "auto")
ridge_reg_b.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %ridge_reg_b.intercept_)
#Coeficientes de regresion
coefs_ridge_b = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": ridge_reg_b.coef_})
coefs_ridge_b

In [ ]:
#Ajustar el modelo ridge con alfa=1000
ridge_reg_c = Ridge(alpha = 1000, solver = "auto")
ridge_reg_c.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %ridge_reg_c.intercept_)
#Coeficientes de regresion
coefs_ridge_c = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": ridge_reg_c.coef_})
coefs_ridge_c

In [ ]:
#Importar clase para la regresión lineal
from sklearn.linear_model import LinearRegression

#Ajustar el modelo
lm = LinearRegression()
lm.fit(predictoras_est, respuesta)

#Obtener error para cada uno
y_pred_lm = lm.predict(predictoras_est)
y_pred_ridge = ridge_reg.predict(predictoras_est)
y_pred_ridge_b = ridge_reg_b.predict(predictoras_est)
y_pred_ridge_c = ridge_reg_c.predict(predictoras_est)

mse_lm = mean_squared_error(respuesta, y_pred_lm)
mse_ridge = mean_squared_error(respuesta, y_pred_ridge)
mse_ridge_b = mean_squared_error(respuesta, y_pred_ridge_b)
mse_ridge_c = mean_squared_error(respuesta, y_pred_ridge_c)
errores_ridge = [mse_lm, mse_ridge, mse_ridge_b, mse_ridge_c]

comparación_ridge = pd.DataFrame({"Modelo": ["Lineal", r"Ridge $\alpha=10$", r"Ridge $\alpha=100$", r"Ridge $\alpha=1000$"], 
                                  "MSE": errores_ridge})

comparación_ridge

El modelo de regresión lineal presenta el menor MSE con 2.6909, lo que indica que es el modelo con mejor capacidad predictiva entre los evaluados. El modelo Ridge con $\alpha=10$ y MSE = 2.7449 muestra un desempeño muy cercano al modelo lineal, lo que sugiere que una regularización leve no afecta significativamente la calidad del ajuste. Sin embargo, los modelos Ridge con MSE = 5.2537 y MSE = 19.1337 presentan un incremento considerable en el error, lo que indica que valores más altos del parámetro de regularización provocan un deterioro en la capacidad predictiva.


In [ ]:
fig = plt.figure(figsize=(6, 6))

#Ancho de barra
barWidth = 0.15

#Definir posicion barras series
r1 = np.arange(len(lm.coef_))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]
r4 = [x + barWidth for x in r3]

#Pintar las barras
plt.bar(r1, lm.coef_, color = "blue", width = barWidth, edgecolor = "white", label = "Lineal")
plt.bar(r2, ridge_reg.coef_, color = "red", width = barWidth, edgecolor = "white", label = "Ridge a")
plt.bar(r3, ridge_reg_b.coef_, color = "green", width = barWidth, edgecolor = "white", label = "Ridge b")
plt.bar(r4, ridge_reg_c.coef_, color = "black", width = barWidth, edgecolor = "white", label = "Ridge c")

plt.xticks([r + barWidth for r in range(len(lm.coef_))], 
[predictoras.columns[i] for i in range(len(predictoras.columns))]);
plt.legend();
plt.xlabel("Variable");
plt.ylabel("Beta");
plt.show(fig)

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score

alphas = np.logspace(-3, 4, 100)
ridge = RidgeCV(alphas=alphas, cv=10)
ridge.fit(predictoras_est, respuesta)
kf = KFold(n_splits=10, shuffle=True, random_state=2026)

mse_mean = list()
mse_std = list()

for alpha in alphas:
    modelo_tmp = RidgeCV(alphas=[alpha], cv=None)  
    scores = cross_val_score(modelo_tmp, predictoras_est, respuesta, scoring='neg_mean_squared_error',cv=kf)
    
    mse = -scores
    mse_mean.append(mse.mean())
    mse_std.append(mse.std())

mse_mean = np.array(mse_mean)
mse_std = np.array(mse_std)

alpha_min = ridge.alpha_


idx_min = np.argmin(mse_mean)
mse_min = mse_mean[idx_min]
se_min = mse_std[idx_min]

idx_1se = np.where(mse_mean <= mse_min + se_min)[0][0]
alpha_1se = alphas[idx_1se]

#Realizar el gráfico
x = np.log(alphas)

plt.figure(figsize=(8, 4))
plt.errorbar(x, mse_mean, yerr=mse_std, fmt='o', color='red', ecolor='gray', elinewidth=0.8, capsize=3, markersize=4)
plt.axvline(np.log(alpha_min), linestyle='--', color="gray")

plt.xlabel("log(alpha)")
plt.ylabel('Mean Squared Error')
plt.title('Validación Cruzada para Ridge')
plt.gca().invert_xaxis()

plt.tight_layout()
plt.show()

print("El valor de alpha que minimiza el mse es %s" %alpha_min )

In [ ]:
#Importar clase
from sklearn.linear_model import Ridge

#Ajustar el modelo
ridge_reg_d = Ridge(alpha = alpha_min, solver = "auto")
ridge_reg_d.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo
# intercepto
print(ridge_reg_d.intercept_)
# coeficientes de regresion
print(ridge_reg_d.coef_)

In [ ]:
#Obtener error para cada uno
y_pred_ridge_d = ridge_reg_d.predict(predictoras_est)

mse_ridge_d = mean_squared_error(respuesta, y_pred_ridge_d)

In [ ]:
mse_ridge_d

In [ ]:
coefs = list()
for a in alphas:
    ridge = linear_model.Ridge(alpha=a, fit_intercept=False)
    ridge.fit(predictoras_est, respuesta)
    coefs.append(ridge.coef_)

In [ ]:
ax = plt.gca()

ax.plot(alphas, coefs)
ax.set_xscale("log")
ax.set_xlim(ax.get_xlim()[::-1])  # reverse axis
plt.xlabel(r"$\alpha$")
plt.ylabel("Coeficientes")
plt.title("Comportamiento de los coeficientes")
plt.axis("tight")
plt.legend([f"Feature {i + 1}" for i in range(predictoras_est.shape[1])], loc="best", fontsize="small")
plt.legend([predictoras.columns[i] for i in range(predictoras_est.shape[1])], loc="best", fontsize="small")
plt.show()

## Ejemplo 1 - Parte 2


Realizar un modelo de regresión Ridge del volúmenes de las ventas con las variables restantes.

a) Con el parámetro de encogimiento $\alpha=0.01$

b) Con el parámetro de encogimiento $\alpha=0.5$

c) Con el parámetro de encogimiento $\alpha=1$

d) Comparar los modelos con el modelo regresión lineal múltiple. ¿Qué se observa?

e) Realizar la validación cruzada para determinar el valor de $\alpha$ que proporciona un menor error cuadrático

f) Comparar el comportamiento de los coeficientes cuando $\alpha$ disminuye y cuando $\alpha$ tiende a hacerse más grande.

In [ ]:
#Importar clase
from sklearn.linear_model import Lasso

#Ajustar el modelo de Lasso con alfa=0.05
lasso_reg = Lasso(alpha = 0.05)
lasso_reg.fit(predictoras_est, respuesta)

#Intercepto
print("El intercepto que se tiene es %s" %lasso_reg.intercept_)
#Coeficientes de regresion
coefs_lasso_a = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": lasso_reg.coef_})
coefs_lasso_a

In [ ]:
#Ajustar el modelo de Lasso con alfa=0.1
lasso_reg_b = Lasso(alpha = 0.1)
lasso_reg_b.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %lasso_reg_b.intercept_)
#Coeficientes de regresion
coefs_lasso_b= pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": lasso_reg_b.coef_})
coefs_lasso_b

In [ ]:
#Ajustar el modelo de Lasso con alfa=1
lasso_reg_c = Lasso(alpha = 1)
lasso_reg_c.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %lasso_reg_c.intercept_)
#Coeficientes de regresion
coefs_ridge_c = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": lasso_reg_c.coef_})
coefs_ridge_c

In [ ]:
#Obtener error para cada uno
y_pred_lasso = lasso_reg.predict(predictoras_est)
y_pred_lasso_b = lasso_reg_b.predict(predictoras_est)
y_pred_lasso_c = lasso_reg_c.predict(predictoras_est)

mse_lasso = mean_squared_error(respuesta, y_pred_lasso)
mse_lasso_b = mean_squared_error(respuesta, y_pred_lasso_b)
mse_lasso_c = mean_squared_error(respuesta, y_pred_lasso_c)
errores_lasso = [mse_lm, mse_lasso, mse_lasso_b, mse_lasso_c]

comparación_lasso = pd.DataFrame({"Modelo": ["Lineal", r"Lasso $\alpha=0.05$", r"Lasso $\alpha=0.1$", r"Lasso $\alpha=1$"], 
                                  "MSE": errores_ridge})

comparación_lasso

El modelo de regresión lineal presenta el menor MSE con 2.6909, lo que indica que es el modelo con mejor capacidad predictiva. El modelo Lasso con $\alpha=0.05$ MSE = 2.7449 muestra un desempeño muy cercano al modelo lineal, lo que sugiere que una penalización leve mantiene la precisión del modelo mientras empieza a simplificarlo.
Sin embargo, los modelos Lasso con MSE = 5.2537 y MSE = 19.1337 presentan un incremento considerable en el error, lo que indica que valores más altos del parámetro de regularización

In [ ]:
from sklearn.linear_model import LassoCV, lasso_path

# Ajustar LassoCV
lasso_cv = LassoCV(cv=10, random_state=2026, n_alphas=100)
lasso_cv.fit(predictoras_est, respuesta)


alphas = lasso_cv.alphas_
mse_path = lasso_cv.mse_path_
mse_mean = mse_path.mean(axis=1)
mse_std = mse_path.std(axis=1)
alpha_min = lasso_cv.alpha_

idx_min = np.argmin(mse_mean)
mse_min = mse_mean[idx_min]
se_min = mse_std[idx_min]
idx_1se = np.where(mse_mean <= mse_min + se_min)[0][0]
alpha_1se = alphas[idx_1se]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(predictoras_est)
alphas_path, coefs_path, _ = lasso_path(X_scaled, respuesta, alphas=alphas)
n_nonzero = np.sum(coefs_path != 0, axis=0)

x = np.log(alphas)

fig, ax = plt.subplots(figsize=(8, 4))

plt.errorbar(x, mse_mean, yerr=mse_std, fmt='o', color='red', ecolor='gray', elinewidth=0.8, capsize=3, markersize=4)


ax.set_xlabel('log(alpha)')
ax.set_ylabel('Mean-Squared Error')
ax.set_title('Lasso Cross-Validation')
ax.invert_xaxis()

plt.axvline(np.log(alpha_min), linestyle='--', color="gray")
plt.tight_layout()
plt.show()

print("El valor de alpha que minimiza el mse es %s" %alpha_min )

In [ ]:
fig = plt.figure(figsize=(6, 6))

#Ancho de barra
barWidth = 0.15

#Definir posicion barras series
r1 = np.arange(len(lm.coef_))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]
r4 = [x + barWidth for x in r3]

#Pintar las barras
plt.bar(r1, lm.coef_, color = "blue", width = barWidth, edgecolor = "white", label = "Lineal")
plt.bar(r2, lasso_reg.coef_, color = "red", width = barWidth, edgecolor = "white", label = "Lasso a")
plt.bar(r3, lasso_reg_b.coef_, color = "green", width = barWidth, edgecolor = "white", label = "Lasso b")
plt.bar(r4, lasso_reg_c.coef_, color = "black", width = barWidth, edgecolor = "white", label = "Lasso c")

plt.xticks([r + barWidth for r in range(len(lm.coef_))], 
[predictoras.columns[i] for i in range(len(predictoras.columns))]);
plt.legend();
plt.xlabel("Variable");
plt.ylabel("Beta");
plt.show(fig)

In [ ]:
coefs = list()
for a in alphas:
    lasso = linear_model.Lasso(alpha=a, fit_intercept=False)
    lasso.fit(predictoras_est, respuesta)
    coefs.append(lasso.coef_)


ax = plt.gca()

ax.plot(alphas, coefs)
ax.set_xscale("log")
ax.set_xlim(ax.get_xlim()[::-1])  # reverse axis
plt.xlabel(r"$\alpha$")
plt.ylabel("Coeficientes")
plt.title("Ridge Coefficients vs Regularization Strength (alpha)")
plt.axis("tight")
plt.legend([f"Feature {i + 1}" for i in range(predictoras_est.shape[1])], loc="best", fontsize="small")
plt.legend([predictoras.columns[i] for i in range(predictoras_est.shape[1])], loc="best", fontsize="small")
plt.show()

## Ejemplo 1 - Parte 3

Realizar un modelo de Red elástica del volúmenes de las ventas con las variables restantes.

a) Fijando $r=0.1$

b) Fijando $r=0.90$ (En ambos casos ajustar por validación cruzada con $k=10$ particiones)

c) Comparar los modelos con el modelo regresión lineal múltiple. ¿Qué se observa?

d) Realizar la validación cruzada para determinar el valor de $\alpha$ y $r$ que proporciona un menor error cuadrático.

In [ ]:
#Importar clase
from sklearn.linear_model import ElasticNetCV

#Ajustar el modelo con r=1 
e_net = ElasticNetCV(cv = 10, l1_ratio = 0.1,random_state=2026)
e_net.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %e_net.intercept_)
#Coeficientes de regresion
coefs_e_net = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": e_net.coef_})
coefs_e_net

In [ ]:
#Ajustar el modelo con r=0.9
e_net_b = ElasticNetCV(cv = 10, l1_ratio = 0.9, random_state=2026)
e_net_b.fit(predictoras_est, respuesta)

#Obtener coeficientes del modelo

#Intercepto
print("El intercepto que se tiene es %s" %e_net_b.intercept_)
#Coeficientes de regresion
coefs_e_net_b = pd.DataFrame({"Variables": predictoras.columns, "Coeficientes": e_net_b.coef_})
coefs_e_net_b

In [ ]:
#Obtener error para cada uno
y_pred_e_net = e_net.predict(predictoras_est)
y_pred_e_net_b = e_net_b.predict(predictoras_est)

mse_e_net = mean_squared_error(respuesta, y_pred_e_net)
mse_e_net_b = mean_squared_error(respuesta, y_pred_e_net_b)

errores_e_net = [mse_lm,mse_e_net, mse_e_net_b]

comparación_e_net= pd.DataFrame({"Modelo": ["Lineal", r"Red elástica $r=0.1$", r"Red elástica $r=0.9$"], 
                                  "MSE": errores_e_net})

comparación_e_net

El modelo de regresión lineal presenta el menor MSE (2.6909), lo que indica que mantiene la mejor capacidad predictiva. El modelo de Red Elástica con $r=0.1$ tiene MSE = 2.7449, muestra un desempeño muy cercano al modelo lineal, lo que sugiere que una combinación adecuada de regularización de Ridge y Lasso permite mantener la precisión mientras se controla la complejidad del modelo.

Sin embargo, el modelo de Red Elástica con $r=0.9$ con MSE = 5.2537 presenta un aumento considerable en el error, indicando que una penalización más fuerte provoca una pérdida de capacidad predictiva.

In [ ]:
fig = plt.figure(figsize=(6, 6))

#Ancho de barra
barWidth = 0.15

#Definir posicion barras series
r1 = np.arange(len(lm.coef_))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]

#Pintar las barras
plt.bar(r1, lm.coef_, color = "green", width = barWidth, edgecolor = "white", label = "Lineal")
plt.bar(r2, e_net.coef_, color = "blue", width = barWidth, edgecolor = "white", label = "Red elástica a")
plt.bar(r3, e_net_b.coef_, color = "red", width = barWidth, edgecolor = "white", label = "Red elástica c")

plt.xticks([r + barWidth for r in range(len(lm.coef_))], 
[predictoras.columns[i] for i in range(len(predictoras.columns))]);
plt.legend();
plt.xlabel("Variable");
plt.ylabel("Beta");
plt.show(fig)

In [ ]:
modelo_cv = ElasticNetCV(alphas=alphas,l1_ratio=np.linspace(0.1, 0.99, 10),  
    cv=10,  max_iter=1000, random_state=2026)

modelo_cv.fit(predictoras_est, respuesta)

print("El valor de alfa que minimiza el error es %s" %modelo_cv.alpha_)
print("El valor de r que minimiza el error es %s" %modelo_cv.l1_ratio_)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.model_selection import KFold, cross_val_score

# Parámetros
r = 0.99

# Modelo CV 
enet_cv = ElasticNetCV( alphas=alphas,l1_ratio=r,cv=10, max_iter=1000, random_state=2026)
enet_cv.fit(predictoras_est, respuesta)
kf = KFold(n_splits=10, shuffle=True, random_state=2026)

mse_mean = list()
mse_std = list()

for alpha in alphas:
    modelo = ElasticNet(alpha=alpha, l1_ratio=r, max_iter=1000)
    scores = cross_val_score(modelo,predictoras_est,respuesta,scoring='neg_mean_squared_error',cv=kf )
    mse = -scores
    mse_mean.append(mse.mean())
    mse_std.append(mse.std())

mse_mean = np.array(mse_mean)
mse_std = np.array(mse_std)

alpha_min = enet_cv.alpha_

idx_min = np.argmin(mse_mean)
mse_min = mse_mean[idx_min]
se_min = mse_std[idx_min]

idx_1se = np.where(mse_mean <= mse_min + se_min)[0][-1]
alpha_1se = alphas[idx_1se]


x = np.log(alphas)

#Realizar el gráfico
plt.figure(figsize=(8, 4))

plt.errorbar( x, mse_mean, yerr=mse_std, fmt='o', ecolor='gray',elinewidth=0.8, capsize=3, markersize=4, color="red")
plt.axvline(np.log(alpha_min), linestyle='--', color='gray')

plt.xlabel("log(alpha)")
plt.ylabel("Mean Squared Error")
plt.title("Validación cruzada red elástica")
plt.gca().invert_xaxis()

plt.tight_layout()
plt.show()

print("alfa_min = ", alpha_min)

In [ ]:
#Ajustar el modelo
e_net_c = ElasticNet(alpha=modelo_cv.alpha_, l1_ratio = modelo_cv.l1_ratio_, random_state=2026)
e_net_c.fit(predictoras_est, respuesta)

y_pred_e_net_c = e_net_c.predict(predictoras_est)
mse_e_net_c = mean_squared_error(respuesta, y_pred_e_net_c)

print("El error cuadrático que se tiene es de %s" %mse_e_net_c)